#### -----------------------------------------------------------------------------<br>Copyright (c) 2024, Lucid Vision Labs, Inc.
##### THE  SOFTWARE  IS  PROVIDED  "AS IS",  WITHOUT  WARRANTY  OF  ANY  KIND,<br>EXPRESS  OR  IMPLIED,  INCLUDING  BUT  NOT  LIMITED  TO  THE  WARRANTIES<br>OF  MERCHANTABILITY,  FITNESS  FOR  A  PARTICULAR  PURPOSE  AND<br>NONINFRINGEMENT.  IN  NO  EVENT  SHALL  THE  AUTHORS  OR  COPYRIGHT  HOLDERS<br>BE  LIABLE  FOR  ANY  CLAIM,  DAMAGES  OR  OTHER  LIABILITY,  WHETHER  IN  AN<br>ACTION  OF  CONTRACT,  TORT  OR  OTHERWISE,  ARISING  FROM,  OUT  OF  OR  IN<br>CONNECTION  WITH  THE  SOFTWARE  OR  THE  USE  OR  OTHER  DEALINGS  IN  THE  SOFTWARE.<br>-----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Warning:
#
# EVS examples support only on windows at the moment
# -----------------------------------------------------------------------------

In [18]:
import time
import locale

from arena_api.system import system
locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')

'en_US.UTF-8'

#### Acquisition: EVS

> This example demonstrates how to acquire images using the EVS
> (Electronic Viewfinder System) stream protocol, which is designed 
> to provide efficient and high-quality image transfer from the camera 
> to the host system.

In [19]:
TAB1 = "  "
TAB2 = "    "
TAB3 = "	 "

In [20]:
"""
This function waits for the user to connect a device before raising an exception
"""

tries = 0
tries_max = 6
sleep_time_secs = 10
while tries < tries_max:  # Wait for device for 60 seconds
    devices = system.create_device()
    if not devices:
        print(
            f'{TAB1}Try {tries+1} of {tries_max}: waiting for {sleep_time_secs} '
            f'secs for a device to be connected!')
        for sec_count in range(sleep_time_secs):
            time.sleep(1)
            print(f'{TAB1}{sec_count + 1 } seconds passed ',
                  '.' * sec_count, end='\r')
            tries += 1
    else:
        print(f'{TAB1}Created {len(devices)} device(s)')
        device = system.select_device(devices)
        nodemap = device.nodemap
        tl_stream_nodemap = device.tl_stream_nodemap
        break
else:
    raise Exception(f'{TAB1}No device found! Please connect a device and run '
                    f'the example again.')


  Created 4 device(s)
  Select device:
    1. ('1c:0f:af:ef:5b:a7', 'ATX051S-C', '', '169.254.168.91')
    2. ('1c:0f:af:c4:07:90', 'TRI02KA-M', '', '169.254.0.41')
    3. ('1c:0f:af:28:cc:b7', 'TRT009S-E', '', '169.254.10.41')
    4. ('1c:0f:af:3f:55:a4', 'PHX064S-M', '', '169.254.20.41')


In [21]:
width = nodemap.get_node("Width").value
height = nodemap.get_node("Height").value
print(f'{TAB1}Image (w, h) = ({width} , {height} )')

  Image (w, h) = (1280 , 720 )


##### Configure device settings

In [22]:
print(f'{TAB1}Set acquisition mode to \'Continuous\'')
initial_acquisition_mode = nodemap.get_node("AcquisitionMode").value
nodemap.get_node("AcquisitionMode").value = "Continuous"

  Set acquisition mode to 'Continuous'


In [23]:
print(f'{TAB1}Set buffer handling mode to \'NewestOnly\'')
tl_stream_nodemap["StreamBufferHandlingMode"].value = "NewestOnly"

  Set buffer handling mode to 'NewestOnly'


#### Acquisition: EVS
>	The EventFormat node determines whether the camera can use the EVS datastream engine. 
>   When set to EVS, Arena switches to the EVS engine. If EVS is not supported, 
>   the acquisition mode is restored to its original setting, and the process is exited.

In [24]:
print(f'{TAB1}Set Event Format to EVT3.0')

try:
    event_format_initial = nodemap.get_node('EventFormat').value
    nodemap["EventFormat"].value = "EVT3_0"
except:
    print(f'{TAB1}Connected camera does not support any EventFormats\n')
    nodemap.get_node("AcquisitionMode").value = initial_acquisition_mode
    system.destroy_device()
    raise

  Set Event Format to EVT3.0


##### Set camera event rate to 10 Mev/s

In [25]:
print(f'{TAB1}Set Camera Event Rate to 10 Mev/s')
erc_enable_initial = nodemap.get_node('ErcEnable').value
nodemap["ErcEnable"].value = True

camera_event_rate_initial = nodemap.get_node('ErcRateLimit').value
nodemap["ErcRateLimit"].value = 10.0

  Set Camera Event Rate to 10 Mev/s


#####  Set evs output format to CDFrame

In [26]:
print(f'{TAB1}Set EVS output format to CDFrame')
tl_stream_nodemap["StreamEvsOutputFormat"].value = "CDFrame"

  Set EVS output format to CDFrame


#####  Set evs accumulation time to auto

In [27]:
print(f'{TAB1}Set EVS Accumulation Time to auto')
stream_frame_generator_fps = tl_stream_nodemap["StreamFrameGeneratorFPS"].value
tl_stream_nodemap["StreamFrameGeneratorAccumTime"].value = int(1000000 / stream_frame_generator_fps)

  Set EVS Accumulation Time to auto


##### EVS Rates Function

Convert an event rate (given in raw events per second) to a more readable string format


In [28]:
def get_evs_event_rate(rate):
    if rate < 1000:
        return f"{rate:.0f} ev/s"
    elif rate < 1000 * 1000:
        return f"{(rate / 1000):.1f} Kev/s"
    elif rate < 1000 * 1000 * 1000:
        return f"{(rate / (1000 * 1000)):.1f} Mev/s"
    else:
        return f"{(rate / (1000 * 1000 * 1000)):.1f} Gev/s"

Convert a gvsp frame rate to a more readable string format

In [29]:
def get_evs_gvsp_frame_rate(rate):
    return f"{rate:.0f} Bid/s"

Convert the link throughput to a more readable string format

In [30]:
def get_evs_link_throughput(rate):
    if rate < 1000:
        return f"{rate:.0f} Bps"
    elif rate < 1000 * 1000:
        return f"{(rate / 1000):.1f} KBps"
    elif rate < 1000 * 1000 * 1000:
        return f"{(rate / (1000 * 1000)):.1f} MBps"
    else:
        return f"{(rate / (1000 * 1000 * 1000)):.1f} GBps"

In [31]:
number_of_buffers = 25

device.start_stream(number_of_buffers)
print(f'{TAB1}Stream started with {number_of_buffers} buffers')

print(f'{TAB1}Get {number_of_buffers} buffers in a list')
buffers = device.get_buffer(number_of_buffers)
print("Success")

'''
Print image buffer info
    Buffers contain image data.
    Image data can also be copied and converted using BufferFactory.
    That is necessary to retain image data, as we must also requeue the buffer.
'''
for count, buffer in enumerate(buffers):
		print(f'{TAB2}buffer{count:{2}} received')

		if buffer.is_incomplete:
			print(f'{TAB3}Image {buffer.frame_id} is incomplete')

		event_rate = tl_stream_nodemap["StreamEvsEventRate"].value
		print(f'{TAB3}Event Rate: {get_evs_event_rate(event_rate)}')

		gvsp_frame_rate = tl_stream_nodemap["StreamEvsGvspFrameRate"].value
		print(f'{TAB3}GVSP Frame Rate: {get_evs_gvsp_frame_rate(gvsp_frame_rate)}')

		link_throughput = tl_stream_nodemap["StreamEvsLinkThroughput"].value
		print(f'{TAB3}GVSP Frame Rate: {get_evs_link_throughput(link_throughput)}')

device.requeue_buffer(buffers)
print(f'{TAB1}Requeued {number_of_buffers} buffers')

device.stop_stream()
print(f'{TAB1}Stream stopped')

  Stream started with 25 buffers
  Get 25 buffers in a list
Success
    buffer 0 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 1 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 2 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 3 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 4 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 5 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 6 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 7 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6250 Bid/s
	 GVSP Frame Rate: 4.7 MBps
    buffer 8 received
	 Event Rate: 86.7 Kev/s
	 GVSP Frame Rate: 6251 Bid/s
	 GVSP Frame Ra

In [32]:
nodemap.get_node("ErcEnable").value = erc_enable_initial
nodemap.get_node("ErcRateLimit").value = camera_event_rate_initial
nodemap.get_node("EventFormat").value = event_format_initial

##### Clean up ----------------------------------------------------------------

> - Destroy device. This call is optional and will automatically be
  called for any remaining devices when the system module is unloading.

In [33]:
nodemap.get_node("AcquisitionMode").value = initial_acquisition_mode

system.destroy_device()
print('Destroyed all created devices')

Destroyed all created devices
